# GeoAI fine-tune (free T4): SFT -> quick eval -> GRPO -> GGUF LoRA

Trains a LoRA adapter for the GeoAI local model **off-machine**; the desktop app only loads the exported
GGUF adapter on top of its base GGUF (llama.cpp). Default run profile: **`qwen3-1.7b`**
(`unsloth/Qwen3-1.7B`, target GGUF `unsloth/Qwen3-1.7B-GGUF` / `Qwen3-1.7B-Q4_K_M.gguf`, thinking off).
Workflow, profiles and the go/no-go rule: `python-backend/core/geoai/finetune/README.md`.

Order: 1 install -> 2 code -> 3 profile -> 4 regenerate + verify data -> 5 SFT -> 6 quick eval -> 7 GRPO
-> 8 export -> 9 download -> 10 base-vs-adapter eval on the desktop.

* Runtime: GPU (Runtime -> Change runtime type -> T4). No secrets are needed: base models are public.
* Times are rough estimates for a ~1.7B model on a T4, not measurements.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install (~3-6 min)
Unsloth brings torch/transformers/trl/peft/bitsandbytes. The GeoAI scorer needs Groundhog + the backend's light deps.

In [ ]:
%%capture
!pip install unsloth
!pip install groundhog fastapi python-multipart
# Optional (faster GRPO rollouts, more VRAM): !pip install vllm   then add --set grpo_use_vllm=true

## 2. Get the GeoCore code
Clone the repo (or upload a zip of `python-backend/` and unzip it to `/content/geocore/python-backend`).

In [ ]:
import os
if not os.path.exists('/content/geocore'):
    !git clone --depth 1 https://github.com/utkarsh-1912/geocore.git /content/geocore
%cd /content/geocore/python-backend

## 3. Run profile
A profile fixes the base model, family preset, thinking switch and the exact base GGUF the desktop runs
(recorded in the adapter sidecar). Profiles: `qwen3-1.7b` (default), `qwen2.5-1.5b`; see `PROFILES` in
`config.py`. Clarification-heavy categories are oversampled (train split / GRPO prompts only); override with
`--set sft_category_weights='{"missing_data": 2.0}'` or `--set sft_category_weights={}` to disable.

In [ ]:
PROFILE = 'qwen3-1.7b'
OUT     = '/content/geoai-ft'
COMMON  = f'--profile {PROFILE} --output-dir {OUT}'

## 4. Regenerate + verify data (CPU, ~3-8 min)
The JSONL data is git-ignored, so regenerate it deterministically. The dry-runs check formatting (incl. multi-step project trajectories), loss masking, held-out leakage, the oversampled category mix and that the reward separates gold from bad completions.

In [ ]:
!python -m core.geoai.training.scaleup --out core/geoai/training/data
!python -m core.geoai.finetune.sft  --dry-run {COMMON}
!python -m core.geoai.finetune.grpo --dry-run {COMMON}

## 5. SFT (LoRA) - estimate 45-90 min on a T4
The log prints the per-category mix before/after oversampling.

In [ ]:
!python -m core.geoai.finetune.sft {COMMON}

## 6. Quick eval on eval_val (base vs SFT) - a few minutes each
Training-time sanity check only. The real go/no-go runs on the desktop with the GGUF and `eval_test`.

In [ ]:
!python -m core.geoai.finetune.evaluate --base {COMMON} --n 40
!python -m core.geoai.finetune.evaluate --adapter-dir {OUT}/sft_adapter {COMMON} --n 40

## 7. GRPO from the SFT adapter - estimate 1-3 h for 300 steps without vLLM
Reward = deterministic GeoAI scorer (`core.geoai.eval.scoring.reward`) + small format reward; prompts use the same category emphasis as SFT. Reduce `grpo_max_steps` for a shorter run.

In [ ]:
!python -m core.geoai.finetune.grpo {COMMON} --set grpo_max_steps=300
!python -m core.geoai.finetune.evaluate --adapter-dir {OUT}/grpo_adapter {COMMON} --n 40

## 8. Export: PEFT adapter -> GGUF LoRA (+ sidecar)
Use the SFT adapter instead if GRPO did not help in step 7. The sidecar pins the profile's target GGUF (repo, file, sha256, fingerprint).

In [ ]:
ADAPTER = f'{OUT}/grpo_adapter'
if not os.path.exists('/content/llama.cpp'):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q /content/llama.cpp/gguf-py
!python -m core.geoai.finetune.export gguf-lora --adapter-dir {ADAPTER} --llama-cpp /content/llama.cpp --outfile {OUT}/export/geoai-lora.gguf

Optional alternative: a merged, quantised q4_k_m GGUF (no LoRA overhead at runtime, ~1.1 GB download). Unsloth builds llama.cpp for this, which takes a while.

In [ ]:
# !python -m core.geoai.finetune.export merged --adapter-dir {ADAPTER} --out-dir {OUT}/export/merged

## 9. Download

In [ ]:
!cd {OUT} && zip -qr /content/geoai-ft-artifacts.zip export sft_adapter/geoai_run.json grpo_adapter/geoai_run.json sft_adapter/geoai_finetune_config.json 2>/dev/null; ls -la /content/geoai-ft-artifacts.zip
from google.colab import files
files.download('/content/geoai-ft-artifacts.zip')

## 10. On the desktop (not here): base vs adapter, same examples, native chat format, thinking off
Put `geoai-lora.gguf` and `geoai-lora.gguf.json` in e.g. `%APPDATA%/GeoCore/models/adapters/` (not the models folder itself).
Leave `GEOAI_THINKING` unset (thinking off is the runtime default).
```
set BASE=%APPDATA%/GeoCore/models/Qwen3-1.7B-Q4_K_M.gguf
python -m core.geoai.finetune.export sidecar --lora geoai-lora.gguf --base-gguf %BASE%
set GEOAI_CHAT_FORMAT=native
python -m core.geoai.eval.runner --provider llama_cpp --model %BASE% --split test --label base --out base.json
set GEOAI_LORA_PATH=C:/path/to/geoai-lora.gguf
python -m core.geoai.eval.runner --provider llama_cpp --model %BASE% --split test --label lora --out lora.json
python -m core.geoai.eval.runner --compare base.json lora.json
python -m core.geoai.finetune.export annotate --lora geoai-lora.gguf --base-results base.json --candidate-results lora.json
```